# Stream A — Full Scale Pipeline (189 subjects)

Optimized for full DAIC-WOZ dataset. Key differences from `02_streamA_improved_pipeline.ipynb`:

| | Improved (dev) | Full Scale (this) |
|--|--|--|
| Speaker filter | Pyannote diarization | Official transcript only (faster) |
| eGeMAPSv02 | Single process | Multiprocess (N_WORKERS cores) |
| Crash recovery | None | Per-subject JSON cache |
| Progress | tqdm | tqdm + log file |

**Estimated runtime**: ~1.5 hrs on 8-core Mac (vs ~7 hrs single-threaded)

---
## Cell 0 — Configuration

In [ ]:
import os
from pathlib import Path

# ============================================================
# ✏️  Edit this section only
# ============================================================
RAW_DATA_DIR  = Path("/Users/Meihui/Downloads/sync/work/whisperx")
LABELS_DIR    = RAW_DATA_DIR
OUTPUT_DIR    = Path("outputs/features")
SEGMENT_DIR   = Path("outputs/segments")
LOG_DIR       = Path("outputs/logs")
CACHE_DIR     = Path("outputs/cache/egemaps")  # per-subject feature cache for crash recovery

# Parallelism — leave 2 cores for system
N_WORKERS = max(1, os.cpu_count() - 2)

# Audio
TARGET_SR           = 16000
MIN_SEGMENT_DUR_SEC = 0.5

# DAIC-WOZ excluded subjects (per official documentation)
EXCLUDED_SUBJECTS = {"342", "394", "398", "460"}

OUTPUT_CSV = OUTPUT_DIR / "full_scale_features.csv"
# ============================================================

for d in [OUTPUT_DIR, SEGMENT_DIR, LOG_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Data dir    : {RAW_DATA_DIR}")
print(f"Output CSV  : {OUTPUT_CSV}")
print(f"Workers     : {N_WORKERS} / {os.cpu_count()} cores")
print(f"Excluded    : {EXCLUDED_SUBJECTS}")

---
## Cell 1 — Environment Check

In [ ]:
import sys, json, logging, warnings, time
from datetime import datetime
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
import opensmile
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

print(f"Python    : {sys.version.split()[0]}")
print(f"PyTorch     : {torch.__version__}")
print(f"OpenSMILE : {opensmile.__version__}")
print(f"NumPy     : {np.__version__}")
print(f"Pandas    : {pd.__version__}")

# Verify eGeMAPSv02 loads correctly
_test_smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals,
)
print(f"eGeMAPSv02: {len(_test_smile.feature_names)} features")

# Logger
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = LOG_DIR / f"full_scale_{RUN_ID}.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_path),
        logging.StreamHandler(sys.stdout),
    ],
)
logger = logging.getLogger("full_scale")
logger.info(f"Run started | run_id={RUN_ID} | workers={N_WORKERS}")
print(f"\nLog file: {log_path}")
print("✅ Environment OK")

---
## Cell 2 — Subject Discovery

In [ ]:
import re

def discover_subjects(raw_dir: Path, excluded: set) -> list:
    """
    Flat directory structure: all XXX_AUDIO.wav and XXX_TRANSCRIPT.csv
    sit directly in raw_dir (no per-subject subfolders).
    subject_id is set to XXX_P to stay compatible with label CSVs.
    """
    subjects = []
    skipped_excluded      = []
    skipped_no_transcript = []

    audio_files = sorted(
        raw_dir.glob("*_AUDIO.wav"),
        key=lambda p: int(re.search(r"(\d+)_AUDIO", p.name).group(1))
    )

    for audio in audio_files:
        m = re.match(r"(\d+)_AUDIO\.wav", audio.name)
        if not m:
            continue
        session_id = m.group(1)         # e.g. '300'
        subject_id = f"{session_id}_P"  # e.g. '300_P' — matches label CSV

        if session_id in excluded:
            skipped_excluded.append(subject_id)
            continue

        transcript = raw_dir / f"{session_id}_TRANSCRIPT.csv"
        if not transcript.exists():
            skipped_no_transcript.append(subject_id)
            transcript = None

        subjects.append({
            "subject_id" : subject_id,
            "session_id" : session_id,
            "audio_path" : audio,
            "transcript" : transcript,
        })

    print(f"Audio files found       : {len(audio_files)}")
    print(f"Subjects ready          : {len(subjects)}")
    print(f"Excluded (official)     : {len(skipped_excluded)}  {skipped_excluded}")
    print(f"No transcript (skipped) : {len(skipped_no_transcript)}  {skipped_no_transcript}")
    return subjects


SUBJECTS = discover_subjects(RAW_DATA_DIR, EXCLUDED_SUBJECTS)
print(f"\nReady to process: {len(SUBJECTS)} subjects")
print(f"Sample: {[s['subject_id'] for s in SUBJECTS[:5]]}")

SUBJECTS = discover_subjects(RAW_DATA_DIR, EXCLUDED_SUBJECTS)
print(f"\nReady to process: {len(SUBJECTS)} subjects")

# # ---- 试跑：只取前10个，确认跑通后删掉这行 ----
# SUBJECTS = SUBJECTS[:10]
# print(f"⚠️  DRY RUN: using first {len(SUBJECTS)} subjects only")

---
## Cell 3 — Load Official Transcripts

Pyannote is skipped — all 189 DAIC-WOZ subjects have official transcripts
with speaker labels, so we use those directly to filter participant speech.
This saves ~8 hours of diarization time.

In [ ]:
def load_transcript(transcript_path: Path) -> list:
    """
    Load DAIC-WOZ official transcript.
    Format: start \t end \t speaker \t text
    Returns participant-only segments with float timestamps.
    """
    df = pd.read_csv(transcript_path, sep="\t",
                     names=["start", "end", "speaker", "text"])
    # Keep participant utterances only, drop scrubbed entries
    participant = df[
        (df["speaker"] == "Participant") &
        (~df["text"].str.contains("scrubbed_entry", na=False))
    ].copy()
    return [
        {"start": float(r.start), "end": float(r.end), "speaker": "PARTICIPANT"}
        for _, r in participant.iterrows()
        if float(r.end) - float(r.start) >= MIN_SEGMENT_DUR_SEC
    ]


ALL_SEGMENTS = {}
n_no_transcript = 0

for subj in tqdm(SUBJECTS, desc="Loading transcripts"):
    sid = subj["subject_id"]
    if subj["transcript"] is not None:
        try:
            segs = load_transcript(subj["transcript"])
            ALL_SEGMENTS[sid] = segs
        except Exception as e:
            logger.error(f"{sid}: transcript load failed — {e}")
            ALL_SEGMENTS[sid] = []
    else:
        # Rare: no official transcript — skip for now
        logger.warning(f"{sid}: no transcript, skipping")
        ALL_SEGMENTS[sid] = []
        n_no_transcript += 1

total_segs   = sum(len(v) for v in ALL_SEGMENTS.values())
total_speech = sum(
    sum(s["end"] - s["start"] for s in segs)
    for segs in ALL_SEGMENTS.values()
)
print(f"\nTranscripts loaded")
print(f"  Total participant segments : {total_segs}")
print(f"  Total speech               : {total_speech/3600:.1f} hrs")
print(f"  Avg per subject            : {total_segs/len(SUBJECTS):.0f} segments")

---
## Cell 4 — Export Participant WAV Segments

In [ ]:
def export_segments(subject_id: str, audio_path: Path,
                    segments: list) -> list:
    """
    Slice audio into per-segment WAV files.
    Applies peak normalization to correct microphone gain differences.
    Returns list of segment dicts with wav_path and duration added.
    """
    out_dir = SEGMENT_DIR / subject_id
    out_dir.mkdir(parents=True, exist_ok=True)

    # Check if already exported
    existing = sorted(out_dir.glob("seg_*.wav"))
    if len(existing) == len(segments):
        # Re-attach paths without re-exporting
        return [
            {**seg, "wav_path": str(existing[i]),
             "duration": seg["end"] - seg["start"]}
            for i, seg in enumerate(segments)
        ]

    audio, _ = librosa.load(str(audio_path), sr=TARGET_SR, mono=True)

    # Peak normalize — fixes microphone gain differences (305_P, 306_P, 307_P)
    peak = np.max(np.abs(audio))
    if peak > 1e-6:
        audio = audio * (0.95 / peak)

    valid = []
    for i, seg in enumerate(segments):
        dur = seg["end"] - seg["start"]
        s   = int(seg["start"] * TARGET_SR)
        e   = min(int(seg["end"] * TARGET_SR), len(audio))
        chunk = audio[s:e]
        if len(chunk) < int(MIN_SEGMENT_DUR_SEC * TARGET_SR):
            continue
        wav_path = out_dir / f"seg_{i:04d}.wav"
        sf.write(str(wav_path), chunk, TARGET_SR)
        valid.append({**seg, "wav_path": str(wav_path), "duration": dur})

    return valid


ALL_EXPORTED = {}
t0 = time.time()

for subj in tqdm(SUBJECTS, desc="Exporting segments"):
    sid  = subj["subject_id"]
    segs = ALL_SEGMENTS.get(sid, [])
    if not segs:
        ALL_EXPORTED[sid] = []
        continue
    try:
        exported = export_segments(sid, subj["audio_path"], segs)
        ALL_EXPORTED[sid] = exported
    except Exception as e:
        logger.error(f"{sid}: export failed — {e}")
        ALL_EXPORTED[sid] = []

elapsed = time.time() - t0
total_exported = sum(len(v) for v in ALL_EXPORTED.values())
print(f"\nExport complete in {elapsed/60:.1f} min")
print(f"  Total segments exported: {total_exported}")

---
## Cell 5 — Parallel eGeMAPSv02 Extraction

Uses `multiprocessing.Pool` to extract features across subjects in parallel.
Each subject's result is cached to JSON — if the run crashes, completed
subjects are skipped on the next run (no re-work).

In [ ]:
from multiprocess import Pool

def extract_subject_egemaps(args: tuple) -> tuple:
    """
    Worker function: extract eGeMAPSv02 for all segments of one subject.
    Initializes its own opensmile instance (required for multiprocessing).
    Loads from cache if already computed.
    """
    sid, segments, cache_dir_str = args
    cache = Path(cache_dir_str) / f"{sid}.json"

    # Return cached result immediately if available
    if cache.exists():
        with open(cache) as f:
            return sid, json.load(f)

    import opensmile as _os
    smile = _os.Smile(
        feature_set=_os.FeatureSet.eGeMAPSv02,
        feature_level=_os.FeatureLevel.Functionals,
    )

    feats_list = []
    for seg in segments:
        try:
            feats = smile.process_file(seg["wav_path"]).iloc[0].to_dict()
            feats["seg_start"]    = seg["start"]
            feats["seg_end"]      = seg["end"]
            feats["seg_duration"] = seg["duration"]
            feats_list.append(feats)
        except Exception as e:
            pass  # log silently, don't crash entire subject

    # Cache result
    with open(cache, "w") as f:
        json.dump(feats_list, f)

    return sid, feats_list


# Build args list — only subjects with exported segments
args_list = [
    (sid, segs, str(CACHE_DIR))
    for sid, segs in ALL_EXPORTED.items()
    if segs
]

# Check how many are already cached
n_cached = sum(1 for sid, _, _ in args_list
               if (CACHE_DIR / f"{sid}.json").exists())
print(f"Subjects to process : {len(args_list)}")
print(f"Already cached      : {n_cached}")
print(f"Need processing     : {len(args_list) - n_cached}")
print(f"Workers             : {N_WORKERS}")
print()

t0 = time.time()
EGEMAPS_FEATURES = {}

with Pool(N_WORKERS) as pool:
    for sid, feats in tqdm(
        pool.imap(extract_subject_egemaps, args_list),
        total=len(args_list),
        desc="eGeMAPSv02"
    ):
        EGEMAPS_FEATURES[sid] = feats

elapsed = time.time() - t0
total_feats = sum(len(v) for v in EGEMAPS_FEATURES.values())
print(f"\neGeMAPSv02 complete in {elapsed/60:.1f} min")
print(f"  Total segments processed : {total_feats}")
print(f"  Subjects with features   : {sum(1 for v in EGEMAPS_FEATURES.values() if v)}")
logger.info(f"eGeMAPSv02 done | {elapsed/60:.1f} min | {total_feats} segments")

---
## Cell 6 — Subject-level Aggregation

In [ ]:
PERCENTILES = [10, 50, 90]

def aggregate(sid: str, feats: list) -> dict:
    """Aggregate segment-level features into one subject row. Stats: mean/std/p10/p50/p90."""
    row = {"subject_id": sid}
    if not feats:
        return row

    numeric_keys = [
        k for k in feats[0]
        if isinstance(feats[0][k], (int, float))
        and k not in ("seg_start", "seg_end")
    ]
    mat = np.array(
        [[f.get(k, np.nan) for k in numeric_keys] for f in feats],
        dtype=float
    )
    for j, key in enumerate(numeric_keys):
        col = mat[:, j]
        col = col[~np.isnan(col)]
        if len(col) == 0:
            for stat in ["mean", "std"] + [f"p{p}" for p in PERCENTILES]:
                row[f"{key}_{stat}"] = np.nan
            continue
        row[f"{key}_mean"] = float(np.mean(col))
        row[f"{key}_std"]  = float(np.std(col))
        for p in PERCENTILES:
            row[f"{key}_p{p}"] = float(np.percentile(col, p))

    row["n_segments"]       = len(feats)
    row["total_speech_sec"] = float(sum(f.get("seg_duration", 0) for f in feats))
    return row


rows = [aggregate(s["subject_id"],
                  EGEMAPS_FEATURES.get(s["subject_id"], []))
        for s in SUBJECTS]

FEATURE_DF = pd.DataFrame(rows)
print(f"Aggregation complete: {FEATURE_DF.shape[0]} subjects x {FEATURE_DF.shape[1]} features")
FEATURE_DF[["subject_id", "n_segments", "total_speech_sec"]].head(10)

---
## Cell 7 — Merge PHQ-8 Labels

In [ ]:
def load_labels(labels_dir: Path) -> pd.DataFrame:
    """Load train + dev split label CSVs and combine."""
    dfs = []
    for fname in ["train_split_Depression_AVEC2017.csv",
                  "dev_split_Depression_AVEC2017.csv"]:
        fpath = labels_dir / fname
        if fpath.exists():
            df = pd.read_csv(fpath)
            df["split"] = fname.split("_")[0]
            dfs.append(df)
            print(f"  Loaded {fname}: {len(df)} subjects")
        else:
            print(f"  NOT FOUND: {fpath}")
    if not dfs:
        return pd.DataFrame()
    labels = pd.concat(dfs, ignore_index=True)
    labels["subject_id"] = labels["Participant_ID"].astype(str) + "_P"
    return labels[["subject_id", "PHQ8_Binary", "PHQ8_Score", "Gender", "split"]]


LABELS_DF = load_labels(LABELS_DIR)

if not LABELS_DF.empty:
    FEATURE_DF = FEATURE_DF.merge(LABELS_DF, on="subject_id", how="left")
    labeled = FEATURE_DF["PHQ8_Score"].notna().sum()
    depressed = (FEATURE_DF["PHQ8_Binary"] == 1).sum()
    normal    = (FEATURE_DF["PHQ8_Binary"] == 0).sum()
    print(f"\nAfter label merge: {FEATURE_DF.shape}")
    print(f"  With PHQ-8 label : {labeled}")
    print(f"  Depressed (>=10) : {depressed}")
    print(f"  Non-depressed    : {normal}")
    print(f"  Missing label    : {len(FEATURE_DF) - labeled}")
else:
    print("Label files not found — place CSVs in:", LABELS_DIR)

---
## Cell 8 — Validation + Save

In [ ]:
# --- Validation ---
print("=" * 55)
print("Validation")
print("=" * 55)

meta_cols = [c for c in ["subject_id","PHQ8_Binary","PHQ8_Score",
                          "Gender","split","n_segments","total_speech_sec"]
             if c in FEATURE_DF.columns]
feat_cols = [c for c in FEATURE_DF.columns if c not in meta_cols]

checks = {
    f"Subjects (expected ~185, got {len(FEATURE_DF)})": len(FEATURE_DF) >= 180,
    f"Feature columns (expected ~440, got {len(feat_cols)})": len(feat_cols) > 400,
    f"No all-NaN rows": not FEATURE_DF[feat_cols].isnull().all(axis=1).any(),
    f"Unique subject per row": FEATURE_DF["subject_id"].nunique() == len(FEATURE_DF),
    f"n_segments > 0 for all": (FEATURE_DF["n_segments"] > 0).all(),
}
all_ok = True
for msg, ok in checks.items():
    print(f"  {'PASS' if ok else 'FAIL'}  {msg}")
    if not ok: all_ok = False

# Flag subjects with very low segment count (possible transcript issue)
low_segs = FEATURE_DF[FEATURE_DF["n_segments"] < 10][["subject_id","n_segments","total_speech_sec"]]
if len(low_segs) > 0:
    print(f"\n  ⚠️  Subjects with <10 segments ({len(low_segs)}):")
    print(low_segs.to_string(index=False))

# --- Save raw features ---
FEATURE_DF.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved: {OUTPUT_CSV.resolve()}")
print(f"  Shape     : {FEATURE_DF.shape}")
print(f"  File size : {OUTPUT_CSV.stat().st_size / 1e6:.1f} MB")
logger.info(f"Feature CSV saved | shape={FEATURE_DF.shape} | path={OUTPUT_CSV}")

---
## Cell 9 — Z-score Normalization + Save

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(FEATURE_DF[feat_cols].values.astype(float))

df_scaled              = FEATURE_DF[meta_cols].copy()
df_scaled[feat_cols]   = X_scaled

# Verify
means = df_scaled[feat_cols].mean()
stds  = df_scaled[feat_cols].std()
print(f"Z-score verification:")
print(f"  Max |mean| : {means.abs().max():.6f}  (should be ~0)")
print(f"  Max std    : {stds.max():.4f}  (should be ~1)")

zscored_csv = OUTPUT_DIR / "full_scale_features_zscored.csv"
df_scaled.to_csv(zscored_csv, index=False)
print(f"\nSaved: {zscored_csv}")
print(f"  Shape     : {df_scaled.shape}")
print(f"  File size : {zscored_csv.stat().st_size / 1e6:.1f} MB")
print("\n✅ Ready for classification. Use: full_scale_features_zscored.csv")
logger.info(f"Z-scored CSV saved | {zscored_csv}")

---
## Cell 10 — QC Summary Plot

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Colour by PHQ-8 group
FEATURE_DF["group"] = FEATURE_DF["PHQ8_Score"].apply(
    lambda x: "depressed" if pd.notna(x) and x >= 10
              else ("normal" if pd.notna(x) else "unknown")
)
GROUP_COLORS = {"normal": "steelblue", "depressed": "tomato", "unknown": "grey"}

QC_FEATURES = [
    ("F0semitoneFrom27.5Hz_sma3nz_amean_mean",      "F0 Mean",              "F0"),
    ("F0semitoneFrom27.5Hz_sma3nz_stddevNorm_mean", "F0 Std Norm (pitch variability)", "F0"),
    ("loudness_sma3_amean_mean",                     "Loudness Mean",        "Energy"),
    ("loudness_sma3_stddevNorm_mean",                "Loudness Std Norm",    "Energy"),
    ("HNRdBACF_sma3nz_amean_mean",                   "HNR (voice quality)",  "Quality"),
    ("jitterLocal_sma3nz_amean_mean",                "Jitter",               "Quality"),
]
available = [(col, label, grp) for col, label, grp in QC_FEATURES
             if col in FEATURE_DF.columns]

subjects    = FEATURE_DF["subject_id"].tolist()
bar_colors  = [GROUP_COLORS[g] for g in FEATURE_DF["group"].tolist()]
x           = np.arange(len(subjects))
SECTION_CLR = {"F0": "#FF8C00", "Energy": "#2196F3", "Quality": "#9C27B0"}

fig, axes = plt.subplots(len(available), 1,
                         figsize=(max(16, len(subjects) * 0.12), len(available) * 3))
if len(available) == 1: axes = [axes]
plt.subplots_adjust(hspace=0.6)

for i, (col, label, grp) in enumerate(available):
    vals       = FEATURE_DF[col].values.astype(float)
    gm, gs     = np.nanmean(vals), np.nanstd(vals)
    is_outlier = np.abs(vals - gm) > 2 * gs
    colors_i   = ["orange" if is_outlier[j] else bar_colors[j] for j in range(len(subjects))]

    axes[i].bar(x, vals, color=colors_i, alpha=0.8, edgecolor="none", linewidth=0)
    for j, (val, out) in enumerate(zip(vals, is_outlier)):
        if out:
            axes[i].text(j, val + 0.03*(np.nanmax(vals)-np.nanmin(vals)),
                         "⚠", ha="center", va="bottom", fontsize=6)
    axes[i].axhline(gm, color="black", linestyle="--", linewidth=0.8,
                    label=f"mean={gm:.3f}")
    axes[i].axhspan(gm-gs, gm+gs, alpha=0.06, color="black")
    axes[i].set_title(f"[{grp}]  {label}", fontsize=9, fontweight="bold",
                      color=SECTION_CLR.get(grp,"black"), loc="left")
    axes[i].set_xticks(x)
    axes[i].set_xticklabels(subjects, rotation=90, fontsize=5)
    axes[i].legend(fontsize=7, loc="upper right")

legend_patches = [
    mpatches.Patch(color="steelblue", label="Normal (PHQ-8 < 10)"),
    mpatches.Patch(color="tomato",    label="Depressed (PHQ-8 ≥ 10)"),
    mpatches.Patch(color="orange",    label="Outlier (>2 SD)"),
    mpatches.Patch(color="grey",      label="Unknown / test split"),
]
fig.legend(handles=legend_patches, loc="upper right",
           bbox_to_anchor=(1.12, 1.0), fontsize=8)
plt.suptitle(
    f"QC: eGeMAPSv02 Key Features — {len(subjects)} subjects\n"
    "Blue=normal  Red=depressed  Orange=outlier  Number=PHQ-8",
    fontsize=11, fontweight="bold", y=1.01
)

qc_path = OUTPUT_DIR / "qc_full_scale_features.png"
plt.savefig(qc_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"QC plot saved: {qc_path}")